# Imipramine CASCI(6e,6o) — build the Hamiltonian and run ADAPT-VQE

This notebook exists because **PySCF publishes no Windows wheel**. Exactly one
step of this workflow needs PySCF — `prepare`, which does the 45-atom RHF/6-31G
and the CASCI(6e,6o) reduction — and it only has to run once. Everything after
it reads a JSON cache.

So: run every cell here, download the two artifacts at the end, and the Windows
machine can do the rest offline forever.

**Open this notebook straight from GitHub** — no download needed:

```
https://colab.research.google.com/github/bhargav2603/fujitsu/blob/main/imipramine_qiskit/colab.ipynb
```

| Cell | Step | Roughly |
|---|---|---|
| 1 | Environment check | seconds |
| 2 | Install the stack | 1–2 min |
| 3 | Clone the repo | seconds |
| 4 | Self-test — prove the stack before spending compute | ~8 min |
| 5 | `prepare` — RHF + CASCI, the only PySCF step | several min |
| 6 | `validate` — exact checks, writes the receipt | 1–2 min |
| 7 | `adapt` — ADAPT-VQE, the published method | minutes |
| 8 | Charts — convergence, energy, why it stopped, what it chose | seconds |
| 9 | Download the artifacts | seconds |

A free CPU runtime is enough. **No GPU is needed and none is used** — at 12
qubits the state vector is 4096 amplitudes (64 KB) and the whole optimization
is sparse linear algebra, so a GPU runtime would only add transfer overhead.

Runtime → Run all works, but read the output of cells 4 and 6 before trusting
anything downstream: both are gates, not formalities.

## 1. Environment check

Nothing here is Colab-specific — the same cells work in any Linux/macOS shell
with the `!` prefixes dropped.

In [ ]:
import platform, sys

print(f"Python   : {platform.python_version()}")
print(f"Platform : {platform.system()} {platform.machine()}")

major, minor = sys.version_info[:2]
if (major, minor) < (3, 10):
    print("\nWARNING: this workflow is developed on 3.12 and needs at least 3.10.")
elif (major, minor) >= (3, 14):
    print("\nWARNING: newer than the tested 3.12; wheels may be missing.")
else:
    print("\nVersion is fine.")

if platform.system() == "Windows":
    raise SystemExit("PySCF has no Windows wheel. That is the whole reason for this notebook.")

## 2. Install

`pyscf` is the one that matters and the one Windows cannot have. `qiskit` and
`qiskit-aer` are only needed for the self-test and for emitting the ADAPT
circuit at the end; the optimization itself runs on SciPy.

**numpy and scipy are deliberately not listed.** Colab already ships versions
that satisfy everything here, and naming them explicitly makes pip resolve them
fresh to the newest release — which upgrades numpy out from under Colab's
preinstalled `numba` and drags `pandas` along with it, producing a wall of
red dependency-conflict text. Letting the dependencies ask for what they need
keeps Colab's own packages intact.

If you *do* see conflicts mentioning `pandas`, `numba`, or `google-colab`, they
are about Colab's preinstalled packages, not this workflow — nothing here
imports any of them. The cell below verifies what actually matters: that the
stack imports and reports its versions. **That, not pip's resolver, is the
test.**

In [ ]:
!pip install -q "pyscf>=2.5" "openfermion>=1.6,<2" "qiskit==2.5.*" "qiskit-aer==0.17.*"

In [ ]:
# Does the stack actually import and work? pip's resolver warnings do not
# answer that; this does.
import importlib
from importlib import metadata

REQUIRED = ["numpy", "scipy", "pyscf", "openfermion", "qiskit", "qiskit_aer"]
DISTRIBUTION = {"qiskit_aer": "qiskit-aer"}

missing, incompatible = [], []
for module in REQUIRED:
    try:
        importlib.import_module(module)
        print(f"  ok        {module:<14} {metadata.version(DISTRIBUTION.get(module, module))}")
    except ModuleNotFoundError as error:
        # Simply not installed. Nothing subtle about it.
        missing.append(module)
        print(f"  MISSING   {module:<14} {error}")
    except Exception as error:
        # Installed but will not load. At this point in a Colab session that
        # almost always means numpy was replaced underneath a C extension
        # compiled against the previous ABI -- which no amount of reinstalling
        # fixes, because the stale module is already in memory.
        incompatible.append(module)
        print(f"  BROKEN    {module:<14} {type(error).__name__}: {error}")

print()
if incompatible:
    print("=" * 70)
    print("Runtime -> Restart session, then re-run THIS cell only (not the")
    print("install cell). A binary extension is holding a numpy that has since")
    print("been replaced; only a restart clears it.")
    print("=" * 70)
elif missing:
    print("=" * 70)
    print(f"Not installed: {', '.join(missing)}")
    print("Re-run the install cell above. If pyscf is the only one missing and")
    print("you are NOT on Colab, note that it has no Windows wheel -- that is")
    print("expected, and is the reason this notebook exists.")
    print("=" * 70)
else:
    import numpy, scipy
    print(f"All {len(REQUIRED)} imports fine. numpy {numpy.__version__} / scipy {scipy.__version__}")
    print()
    print("Dependency-conflict warnings naming pandas, numba or google-colab are")
    print("about Colab's own preinstalled packages and do not affect this")
    print("workflow -- nothing in it imports any of them. This table is the test.")

## 3. Get the workflow

Cloning from GitHub is the easy path — nothing to upload, and re-running the cell
picks up any changes you have pushed since.

Set `REPO` to your own fork if you are not running from the original. If the
repository is **private**, use a personal access token:
`https://<TOKEN>@github.com/owner/repo.git`.

The zip fallback below is there for the offline case.

In [ ]:
import os, shutil, subprocess
from pathlib import Path

REPO = "https://github.com/bhargav2603/fujitsu.git"
BRANCH = "main"
FOLDER = "imipramine_qiskit"   # the workflow folder inside the repo

target = Path("/content/repo")
if target.exists():
    shutil.rmtree(target)      # always start from a clean clone
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, str(target)],
    check=True,
)

# Find run.py rather than assuming the layout, so a renamed or nested folder
# still works.
candidates = sorted(target.rglob("run.py"))
matching = [p for p in candidates if p.parent.name == FOLDER] or candidates
if not matching:
    raise SystemExit(f"No run.py found in {REPO}")
ROOT = matching[0].parent
os.chdir(ROOT)

commit = subprocess.run(
    ["git", "-C", str(target), "log", "-1", "--format=%h %ad %s", "--date=short"],
    capture_output=True, text=True,
).stdout.strip()
print(f"Working in : {ROOT}")
print(f"Commit     : {commit}\n")

required = [
    "molecule.py", "hamiltonian.py", "validation.py",
    "qiskit_runtime.py", "adapt_runtime.py", "selftest.py", "summary.py",
]
missing = [name for name in required if not (ROOT / name).is_file()]
print("Missing:", missing if missing else "nothing")

In [ ]:
# FALLBACK ONLY -- skip this cell if the clone above worked.
# Use it when the repo is private and you would rather not paste a token:
# zip the folder locally and upload it here.
#
#   PowerShell:  Compress-Archive -Path imipramine_qiskit -DestinationPath wf.zip -Force
#
# import os, zipfile
# from pathlib import Path
# from google.colab import files
#
# uploaded = files.upload()
# for name in uploaded:
#     if name.endswith(".zip"):
#         with zipfile.ZipFile(name) as archive:
#             archive.extractall("/content/workflow")
# candidates = sorted(Path("/content/workflow").rglob("run.py"))
# if not candidates:
#     raise SystemExit("No run.py in the upload -- did you zip the folder itself?")
# os.chdir(candidates[0].parent)
# print("Working in", candidates[0].parent)

## 4. Prove the stack before spending compute on chemistry

This needs no PySCF and no cache. It generates a random but structurally real
CAS(6e,6o) Hamiltonian and checks the operator conversion, the qubit ordering,
the gradients, a full VQE and the whole ADAPT path against exact classical
answers.

**If this is not green, stop.** A mapping or ordering bug that survives to
`prepare` shows up as a plausible wrong number, not as an error.

In [ ]:
!python run.py selftest

## 5. Build the Hamiltonian

`--diagnose` runs RHF once and prints every orbital either side of the gap with
its Mulliken population on the aromatic π core, without building anything.

This matters more than it sounds. CAS(6e,6o) on canonical orbitals takes
HOMO−2…LUMO+2, and for imipramine those *should* be the dibenzazepine π system.
They could instead be the side-chain dimethylamino lone pair or the propyl tail
— irrelevant to the pharmacophore, and a run built on them would still converge
cleanly and still pass every mapping check. `prepare` refuses to build if an
active orbital falls short of its localization threshold.

In [ ]:
!python run.py prepare --diagnose

In [ ]:
!python run.py prepare

## 6. Validate — this is what writes the receipt

Exact dense diagonalization of the 4096×4096 Hamiltonian, per penalty setting.
`vqe` and `adapt` both refuse to run without the receipt this produces.

**Keep `0` in the list.** Every ADAPT pool operator commutes with N and
S<sub>z</sub> exactly, so the state cannot leave the six-electron sector and no
penalty is applied — the unconstrained entry is the one `adapt` needs. `1` and
`4` are there for the hardware-efficient `vqe`, which does leak and does need a
penalty.

Read the per-penalty verdict table before moving on. A penalty marked
`REJECTED` does not select the six-electron sector, and a VQE run against it
would faithfully converge to the wrong answer — use one marked `ok`.

Do not continue unless this prints `ALL CHECKS PASSED`.

In [ ]:
!python run.py validate --number-penalty 0 1 4

## 7. ADAPT-VQE — the published method

The ansatz is grown one operator at a time from a UCCGSD pool, every parameter
re-optimized at each step, stopping when no operator in the pool has a gradient
worth adding. The Qiskit circuit is emitted at the end and its energy checked
against the sparse algebra that produced it.

Read three lines of the output:

* **`Error vs CASCI`** — the benchmark. The paper reports 1.15 mHa against *its*
  CASCI; this is the directly comparable quantity, and 1.6 mHa is the pass mark.
* **`Circuit 2q gates`** — compare with their 244 two-qubit gates.
* **`Circuit vs algebra`** — must say `exact synthesis`.

The total energy is **not** comparable with theirs: they used an unpublished
conformer ensemble and this uses one PubChem conformer, which is worth tens of
mHa of real physics. `adapt` prints that warning itself.

**Expect a ~2 minute pause** after the pool line before the iteration table
starts. The `uccgsd` pool is 435 operators and each one has its sparse matrix
built once at startup (~570 MB peak). That is exactly what makes operator
selection afterwards cost 48 ms for the *entire* pool. `--pool uccsd` is a
seventh of the size if you want a quicker first look.

In [ ]:
!python run.py adapt --pool uccgsd

In [ ]:
# Optional: the smaller pool, and the hardware-efficient ansatz for contrast.
# The HEA needs a number penalty -- use one the table in cell 6 marked `ok`,
# not necessarily 1.
!python run.py adapt --pool uccsd
!python run.py vqe --method statevector --layers 4 --number-penalty 1
!python run.py summary

## 8. See the result

Four questions the numbers answer slowly and a picture answers at a glance:

1. **Did it converge, and to what?** Error against CASCI per operator added, on a
   log axis, with the 1.6 mHa pass mark and the published 1.15 mHa drawn in.
   This is the chart the whole workflow exists to produce.
2. **Where did the energy actually go?** The same run in absolute hartree,
   between the Hartree–Fock and CASCI lines, so the correlation energy being
   recovered is a visible distance rather than a number to be trusted.
3. **Why did it stop?** The largest remaining pool gradient against the threshold
   that ends the loop. A run that hit the operator cap looks nothing like one
   that converged — and that distinction matters more than the final energy,
   because a capped run is not a result, it is an unfinished one.
4. **What did it choose?** The selected excitations by rotation angle, split into
   singles and doubles.

Then one leaderboard across every run in `results/`, so pools and ansätze are
compared on the axis that matters rather than by scrolling back through output.

`visualize.show()` renders inline. `!python run.py plot` writes the same charts
as PNGs if you want them in a document.

In [ ]:
%matplotlib inline
import importlib

import visualize

# Re-import so an edit to visualize.py takes effect without restarting Colab.
importlib.reload(visualize)

visualize.show("results")

## 9. Download the artifacts

Two files are the point of this notebook. Put them next to `run.py` on the
Windows machine and every remaining command works there natively.

The cache carries its own specification hash and the receipt is bound to the
cache, the molecule spec and the physics modules, so a stale or mismatched file
is a hard error rather than a wrong answer. Copying them between machines is
safe by construction.

`results.zip` carries the run outputs and, if you ran `run.py plot`, the PNGs.

In [ ]:
import shutil
from pathlib import Path

from google.colab import files

# Save the charts alongside the JSON so they travel in the same zip.
!python run.py plot

for name in ("hamiltonian_cas6e6o.json", "hamiltonian_cas6e6o.validated.json"):
    path = Path(name)
    print(f"{name:<40} {path.stat().st_size/1e6:.2f} MB" if path.is_file() else f"{name:<40} MISSING")
    if path.is_file():
        files.download(name)

if Path("results").is_dir():
    shutil.make_archive("results", "zip", "results")
    files.download("results.zip")